In [5]:
# %pip install bertopic huggingface_hub scikit-learn
# %pip install --upgrade llama-cpp-python

import pandas as pd
from bertopic import BERTopic
from bertopic.representation import LlamaCPP
from llama_cpp import Llama
from huggingface_hub import hf_hub_download

In [6]:
PARQUET_PATH = r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260310_180458_ADS_Curation_Run\data\publications_disambiguated.parquet"

df = pd.read_parquet(PARQUET_PATH)
docs = df["full_text"].dropna().astype(str).str.strip()
docs = docs[docs != ""].tolist()

# Optional: Für den MWE-Test auf 500 Dokumente begrenzt. Ggf. anpassen.
docs = docs[:500]
print(f"Loaded {len(docs)} documents")

Loaded 382 documents


In [7]:
model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
)

llm = Llama(
    model_path=model_path,
    n_gpu_layers=0,  # 0 = CPU only
    n_ctx=4096,     # Grosses Kontextfenster für lange Texte
    stop="Q:",       # Trigger für das Ende des statischen BERTopic Prompts
    verbose=False,
)

representation_model = LlamaCPP(llm)

llama_context: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [8]:
topic_model = BERTopic(
    representation_model=representation_model,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

topic_model.get_topic_info()

2026-03-11 14:44:06,931 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2026-03-11 14:44:14,811 - BERTopic - Embedding - Completed ✓
2026-03-11 14:44:14,814 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 14:44:15,331 - BERTopic - Dimensionality - Completed ✓
2026-03-11 14:44:15,333 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 14:44:15,355 - BERTopic - Cluster - Completed ✓
2026-03-11 14:44:15,362 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 7/7 [02:02<00:00, 17.55s/it]
2026-03-11 14:46:18,303 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,82,-1_Environmental impacts of eating meat___,"[Environmental impacts of eating meat, , , , ,...",[Bohr's and Mach's Conceptions of Non-Locality...
1,0,165,0_Environmental impacts of eating meat___,"[Environmental impacts of eating meat, , , , ,...",[On general-relativistic and gauge field theor...
2,1,66,1_Environmental impacts of eating meat___,"[Environmental impacts of eating meat, , , , ,...",[Hans Ertel and cosmology. This paper deals wi...
3,2,25,2_Environmental impacts of eating meat___,"[Environmental impacts of eating meat, , , , ,...",[On the General-Relativistic and Covariant For...
4,3,17,3_Environmental impacts of eating meat___,"[Environmental impacts of eating meat, , , , ,...",[Note on the estimating the Sun's radiation ou...
5,4,14,4_Environmental impacts of eating meat___,"[Environmental impacts of eating meat, , , , ,...",[The Elementary Constant and the Mathematics. ...
6,5,13,5_Environmental impacts of eating meat___,"[Environmental impacts of eating meat, , , , ,...",[The relativistic enlargement of the apparent ...
